# spheroid-seg — Colab GPU training

This notebook takes a fresh Colab runtime to a verified training result for
the v0.1 JAX/Flax U-Net. It works on both GPU and CPU runtimes: on GPU it
runs the `configs/base.yaml` overfit-one-batch acceptance check and a short
full-training run with `configs/colab.yaml` (same model as `base.yaml` but
batch_size 4 to fit a ~15 GiB Colab T4). On CPU it falls back to a short,
CPU-friendly smoke run.

**Before you start:** in Colab, go to *Runtime → Change runtime type* and
select a GPU (a T4 is enough). Then run the cells in order. Leave
`USE_DRIVE_DATA = False` to use the built-in synthetic fallback; set it to
`True` to load a private `data/` directory from Google Drive.


In [ ]:
# Setup constants, detect GPU, and define the streaming command helper.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path(os.environ.get("SPHEROID_SEG_REPO", "/content/spheroid-seg"))
HAS_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
UV_EXTRA = "cuda12" if HAS_GPU else ""

# Google Drive data loading (maintainer only; external users leave this False).
USE_DRIVE_DATA = False
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/spheroid-seg/data")

# Training config used below for the short full-training sanity check.
TRAIN_CONFIG = "configs/colab.yaml" if HAS_GPU else "configs/tiny.yaml"
TRAIN_PREFIX = Path(TRAIN_CONFIG).stem + "_"


def run(cmd, cwd=None, env=None):
    """Run cmd, streaming its output live into the cell; raise on failure."""
    proc_env = os.environ.copy()
    if env:
        proc_env.update(env)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=proc_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")


print(f"REPO={REPO}")
print(f"HAS_GPU={HAS_GPU}")
print(f"UV_EXTRA={UV_EXTRA!r}")
print(f"USE_DRIVE_DATA={USE_DRIVE_DATA}")
print(f"TRAIN_CONFIG={TRAIN_CONFIG}")

## Clone the repository

The notebook drives everything through subprocesses inside the cloned repo;
the Colab kernel itself stays stock. Every command below passes the repo
directory explicitly so cell ordering is the only ordering assumption.

In [ ]:
if REPO.exists():
    print(f"{REPO} already exists; skipping clone.")
else:
    REPO.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "https://github.com/edgardomarchi/spheroid-seg.git", str(REPO)])

## Install uv

Bootstrap uv into the kernel environment so that plain `uv` is on PATH for
all later subprocess calls.

In [ ]:
if shutil.which("uv"):
    print("uv already on PATH; skipping pip install.")
else:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

## Sync the project

On GPU we add the CUDA 12 JAX extra; on CPU we sync the base project only.
The dev group is included so the smoke test can run.

In [ ]:
cmd = ["uv", "sync", "--group", "dev", "--group", "notebooks"]
if UV_EXTRA:
    cmd.extend(["--extra", UV_EXTRA])
run(cmd, cwd=REPO)

## Verify JAX sees the device

Expected output on GPU: a list containing one `CudaDevice`. On CPU: a list
containing one `CpuDevice`.

In [ ]:
run(["uv", "run", "python", "-c", "import jax; print(jax.devices())"], cwd=REPO)

## Optional: load private data from Google Drive

This cell is for the maintainer's private annotated data. External users should
leave `USE_DRIVE_DATA = False` at the top of the notebook; the notebook then
trains on the built-in synthetic fallback and nothing is uploaded or committed.

When `USE_DRIVE_DATA = True`, Drive is mounted and the `raw/`, `masks/`, and
`splits/` folders are copied into the repo's `data/` directory. Paths with
spaces are handled with `shutil.copytree`, not shell commands. `data/` is
`.gitignore`d (except `data/splits/*.txt`), so no images are committed.


In [ ]:
if USE_DRIVE_DATA:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_data_dir = Path(DRIVE_DATA_DIR)
    for sub in ["raw", "masks", "splits"]:
        src = drive_data_dir / sub
        dst = REPO / "data" / sub
        shutil.copytree(src, dst, dirs_exist_ok=True)

    n_raw = len(list((REPO / "data" / "raw").glob("*")))
    n_masks = len(list((REPO / "data" / "masks").glob("*")))
    print(f"Drive data copied: raw={n_raw}, masks={n_masks}")
    if n_raw == 0 or n_masks == 0:
        print(
            "WARNING: no real image/mask pairs found -- training will fall back to "
            "the synthetic dataset. Check DRIVE_DATA_DIR."
        )
else:
    print("USE_DRIVE_DATA=False: training will use the synthetic fallback dataset.")

## Clean-room smoke test

Run the test suite. This doubles as the design-doc §5 clean-room
reproducibility check on a fresh machine.

In [ ]:
run(["uv", "run", "pytest", "-q"], cwd=REPO)

## Pending acceptance check: overfit-one-batch

On a GPU the full `configs/base.yaml` check should run and the loss should
fall monotonically to near-zero, showing that the 512² / 7.7M-param model
can memorize a single batch. On CPU this cell is skipped because it is
impractical on CPU.

In [ ]:
if HAS_GPU:
    run(
        [
            "uv",
            "run",
            "python",
            "-m",
            "spheroid_seg.train",
            "--config",
            "configs/base.yaml",
            "--overfit-one-batch",
        ],
        cwd=REPO,
        env={"XLA_PYTHON_CLIENT_MEM_FRACTION": "0.9"},
    )
else:
    print("No GPU detected: skipping base.yaml --overfit-one-batch (impractical on CPU).")

## Short full-training sanity check

On GPU we run a few epochs of `configs/colab.yaml` (same model as
`configs/base.yaml` but batch_size 4 for ~15 GiB Colab GPUs) on the
synthetic fallback to measure real GPU speed. On CPU we run
`configs/tiny.yaml` for a fast smoke run. Set `USE_DRIVE_DATA = True`
in the setup cell above to copy a private `data/` directory from Drive
before training.

In [ ]:
if HAS_GPU:
    epochs = 3
else:
    epochs = 2
    print(f"No GPU detected: running CPU-friendly training ({TRAIN_CONFIG}, {epochs} epochs).")

run(
    [
        "uv",
        "run",
        "python",
        "-m",
        "spheroid_seg.train",
        "--config",
        TRAIN_CONFIG,
        "--epochs",
        str(epochs),
    ],
    cwd=REPO,
    env={"XLA_PYTHON_CLIENT_MEM_FRACTION": "0.9"},
)

## Training curves

Plot train/val loss and per-class Dice from the latest run's CSV log.
Good curves: loss falls smoothly, aggregate (class 2) and loose-cell
(class 1) Dice rise toward ≥0.9 on the trivial synthetic task; see
`docs/training.md` for the full sanity-check discussion.

In [ ]:
import csv

import matplotlib.pyplot as plt

run_dirs = sorted((REPO / "outputs" / "runs").glob(f"{TRAIN_PREFIX}*"))
if not run_dirs:
    raise RuntimeError("No training run found in outputs/runs/")
latest_run = run_dirs[-1]
log_path = latest_run / "logs" / "train_log.csv"

epochs = []
train_loss = []
val_loss = []
dice_classes = {}
with log_path.open("r", newline="") as f:
    reader = csv.DictReader(f)
    dice_fieldnames = [n for n in reader.fieldnames if n.startswith("dice_class_")]
    for row in reader:
        epochs.append(int(row["epoch"]))
        train_loss.append(float(row["train_loss"]))
        val_loss.append(float(row["val_loss"]))
        for name in dice_fieldnames:
            dice_classes.setdefault(name, []).append(float(row[name]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_loss, label="train loss")
axes[0].plot(epochs, val_loss, label="val loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Train / validation loss")
axes[0].legend()
axes[0].grid(True)

for name, values in dice_classes.items():
    cls = name.split("_")[-1]
    axes[1].plot(epochs, values, label=f"class {cls}")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("Dice")
axes[1].set_title("Per-class validation Dice")
axes[1].legend()
axes[1].grid(True)

fig.tight_layout()
plt.show()
print(f"Plotted training curves from {log_path}")

## Download checkpoints before the session dies

Colab sessions can be cut at any time. Zip the latest run and download it.
Outside Colab the archive is created but the download step is skipped.

In [ ]:
try:
    from google.colab import files

    has_colab = True
except ModuleNotFoundError:
    has_colab = False

run_dirs = sorted((REPO / "outputs" / "runs").glob(f"{TRAIN_PREFIX}*"))
if not run_dirs:
    raise RuntimeError("No training run found in outputs/runs/")
latest_run = run_dirs[-1]
archive = REPO.parent / "spheroid-seg-checkpoints.zip"
run(["zip", "-r", str(archive), str(latest_run)], cwd=REPO)
if has_colab:
    files.download(str(archive))
else:
    print(f"Checkpoint archive created at {archive}")